# welded_fatigue_v2 - openBIS provisioning notebook (FB 7.2, S-N tests on welded joints)

PRD: `BAM_PRD_Workflow_FB72_v2.md` | design: `generated/welded_fatigue_v2/design.json` | openBIS 20.10 | pyBIS 1.37.x | generated 2026-10-06

## Flat structure (one level below the specimen)

```
/FB_72 (space)
└── /FB_72/WELDED_FATIGUE (project)
    ├── WORKFLOW_PLAN                  GENERAL_PROTOCOL (PROCEDURE = step -> location table)
    ├── SERIES_1_BUTT_WELD             COLLECTION ($DEFAULT_OBJECT_TYPE = SAMPLE.WELDED_FATIGUE_SPECIMEN)
    │   ├── S1B-NN                     SAMPLE.WELDED_FATIGUE_SPECIMEN  (00, 01, 02, 04, 05, 06, Next step)
    │   │   ├── S1B-NN-SCAN            EXPERIMENTAL_STEP              (03-04)  <- ZEISS GOM ATOS 5
    │   │   ├── S1B-NN-TEST            EXPERIMENTAL_STEP.MECHANICAL_TEST.CYCLIC_FATIGUE_TEST (07-11)
    │   │   │                          <- strain gauge, adhesive, crack-lum. set, Luxonis camera, IMC amplifier, servo-hydraulic rig
    │   │   └── S1B-NN-FRACTURE        EXPERIMENTAL_STEP              (12)     <- macro camera, digital microscope
    └── SERIES_1_CRUCIFORM_WELD        COLLECTION (same pattern, codes S1C-NN)
/MATERIALS/7_2_* (inventory, created once): instruments, testing machine, auxiliary materials
```

Steps are children of the specimen only (depth 1), live in the series collection, no step->step links; inventory objects are
**parents** of the steps. The field **Next step** (`FATIGUE.NEXT_STEP`) on every specimen shows what has to happen next.

## Two paths

| Path | Tool | When |
|---|---|---|
| **Setup / registration** (this notebook) | pyBIS | once: collections, inventory, workflow plan; then per new specimen: `register_specimen()` + `start_step()` |
| **Per-run ingestion** | parser (`/bam-generate-parser`, `generated/welded_fatigue_v2/parsers/`) | after each test run: read IMC/GOM files, fill step properties, upload datasets |

## Masterdata is NOT created here

Object types, property types and vocabularies (2 new types, 33 property types, 6 new + 2 extended vocabularies) come from
**bam-masterdata** (admin sync after the PR is merged) or from `tools/openbis_apply.js` (idempotent apply in a signed-in ELN tab).
This notebook only **checks** that they exist (section 3) and stops with a clear message if anything is missing.
Everything below is idempotent: existing objects (by identifier) are reused and never overwritten.

## 1. Configuration and authentication

No credentials are stored in this notebook. Either export a personal access token (PAT) as environment variable
`OPENBIS_TOKEN` before starting Jupyter, or leave it unset: you are then asked for username/password **once**, a PAT is
created and the session continues with that PAT. Store the PAT in a password manager / `OPENBIS_TOKEN` for later runs.

In [ ]:
import getpass
import os
from datetime import datetime
from pathlib import Path

from pybis import Openbis

OPENBIS_URL = "https://dev1.datastore.bam.de"   # TODO: production URL when going live
SPACE = "FB_72"
PROJECT = "WELDED_FATIGUE"
PROJECT_ID = f"/{SPACE}/{PROJECT}"
SESSION_NAME = "welded_fatigue_v2-provisioning"

# repo root (works when started from the repo root or from notebooks/)
REPO_ROOT = Path.cwd() if (Path.cwd() / "generated").exists() else Path.cwd().parent
DESIGN_DIR = REPO_ROOT / "generated" / "welded_fatigue_v2"

In [ ]:
o = Openbis(OPENBIS_URL, verify_certificates=True)
_env_token = os.environ.get("OPENBIS_TOKEN")
if _env_token:
    o.set_token(_env_token, save_token=True)
else:
    # first run only: log in once, obtain a PAT, continue with the PAT
    _user = input("openBIS username: ")
    o.login(_user, getpass.getpass("openBIS password (used once to obtain a PAT): "), save_token=True)
    _pat = o.get_or_create_personal_access_token(sessionName=SESSION_NAME)
    _pat = getattr(_pat, "permId", _pat)
    o.logout()
    o = Openbis(OPENBIS_URL, verify_certificates=True)
    o.set_token(_pat, save_token=True)
    print(f"PAT in use: {str(_pat)[:8]}...  -> store it (password manager / env OPENBIS_TOKEN) for later runs.")
    del _pat

assert o.is_session_active(), "Session not active - check OPENBIS_TOKEN or re-run this cell"
print("Connected to", OPENBIS_URL)

## 2. Idempotent helpers

* `ensure_collection()` - get-or-create a `COLLECTION` (experiment) with `$NAME` / `$DEFAULT_OBJECT_TYPE`.
  Projects are **not** created by this notebook (no project creation in the allowed pyBIS API set): `/FB_72/WELDED_FATIGUE`
  must exist (ELN UI, or `tools/openbis_demo.js`). A missing project shows up as a clear error here.
* `ensure_sample()` - get-or-create an object by identifier. Existing objects are returned unchanged (type mismatch is reported).
  Properties that are not assigned to the type on the instance are dropped and listed (same behaviour as `tools/openbis_demo.js`).

In [ ]:
def _get_or_none(getter, identifier):
    """pyBIS raises ValueError for 'not found'; anything else (auth, network) propagates."""
    try:
        return getter(identifier)
    except ValueError:
        return None


_ASSIGNED = {}


def assigned_props(sample_type):
    """Lower-case codes of the properties assigned to a sample type (unsaved probe object, nothing is written)."""
    if sample_type not in _ASSIGNED:
        probe = o.new_sample(type=sample_type, space=SPACE)
        _ASSIGNED[sample_type] = {k.lower() for k in probe.props.all()}
    return _ASSIGNED[sample_type]


def clean_props(sample_type, props, label=""):
    """Drop None values and properties not assigned on the instance (reported, not fatal)."""
    have = assigned_props(sample_type)
    out, dropped = {}, []
    for k, v in (props or {}).items():
        if v is None:
            continue
        if k.lower() in have:
            out[k.lower()] = v
        else:
            dropped.append(k)
    if dropped:
        print(f"  ! {label or sample_type}: not assigned on instance, dropped: {', '.join(dropped)}")
    return out


def ensure_collection(project_id, code, name, default_object_type, create=True):
    ident = f"{project_id}/{code}"
    exp = _get_or_none(o.get_experiment, ident)
    if exp is not None:
        print(f"  = {ident} (exists)")
        return exp
    if not create:
        raise RuntimeError(f"Collection {ident} is missing - ask the inventory owner to create it (not created by this notebook).")
    exp = o.new_experiment(code=code, type="COLLECTION", project=project_id,
                           props={"$name": name, "$default_object_type": default_object_type})
    try:
        exp.save()
    except Exception as e:
        raise RuntimeError(f"Could not create {ident}. Does project {project_id} exist? "
                           f"Projects are not created by this notebook. ({e})") from e
    print(f"  + {ident}")
    return exp


def ensure_sample(identifier, sample_type, experiment=None, parents=None, props=None):
    """Get-or-create by identifier '/SPACE/PROJECT/CODE'. Existing objects are not modified."""
    s = _get_or_none(o.get_sample, identifier)
    if s is not None:
        have_type = getattr(s.type, "code", s.type)
        if have_type != sample_type:
            print(f"  ! {identifier} exists with type {have_type} (design: {sample_type}) - left unchanged, migrate manually")
        else:
            print(f"  = {identifier} (exists)")
        return s
    _, space, project, code = identifier.split("/")
    s = o.new_sample(type=sample_type, space=space, project=f"/{space}/{project}", experiment=experiment,
                     code=code, parents=parents or [], props=clean_props(sample_type, props, code))
    s.save()
    print(f"  + {identifier} ({sample_type})")
    return s

## 3. Masterdata check (read-only, stops if anything is missing)

For every type an **unsaved** probe object is built (`o.new_sample(...)` without `.save()`): pyBIS then loads the type and its
property assignments, and validates vocabulary terms when a controlled-vocabulary property is set. Nothing is written.
Missing items must be provisioned first: bam-masterdata sync (admin) or `tools/openbis_apply.js` with `generated/welded_fatigue_v2/spec.json`.

In [ ]:
REQUIRED_TYPES = {
    # new types (CREATE) -> all design properties must be assigned
    'SAMPLE.WELDED_FATIGUE_SPECIMEN': [
        'FATIGUE.NEXT_STEP',
        'FATIGUE.MANUFACTURER_DESIGNATION',
        'FATIGUE.SOURCE_SHEET',
        'EXPERIMENTAL_STEP.WELDMENT_TYPE',
        'FATIGUE.TARGET_QUALITY_LEVEL',
        'FATIGUE.INITIAL_ANGULAR_DISTORTION',
        'FATIGUE.INITIAL_EDGE_MISALIGNMENT',
        'FATIGUE.WAS_STRAIGHTENED',
        'FATIGUE.ANGULAR_DISTORTION_AFTER_STRAIGHTENING',
        'FATIGUE.EDGE_MISALIGNMENT_AFTER_STRAIGHTENING',
        'DEFECT_DESCRIPTION',
        'FATIGUE.WELD_POST_TREATMENT_NOTES',
        'FATIGUE.ISO_5817_ANNEX_C_RATING',
        'FATIGUE.LOAD_LEVEL',
        'FATIGUE.TEST_ORDER',
        'FATIGUE.THICKNESS_POS_1',
        'FATIGUE.THICKNESS_POS_2',
        'FATIGUE.THICKNESS_POS_3',
        'FATIGUE.WIDTH_POS_1',
        'FATIGUE.WIDTH_POS_2',
        'FATIGUE.WIDTH_POS_3',
        'FATIGUE.MEAN_THICKNESS',
        'FATIGUE.MEAN_WIDTH',
    ],
    'EXPERIMENTAL_STEP.MECHANICAL_TEST.CYCLIC_FATIGUE_TEST': [
        'FATIGUE.STRAIN_GAUGE_DISTANCE_TO_WELD_TOE',
        'FATIGUE.STRAIN_GAUGE_DISTANCE_TO_EDGE',
        'FATIGUE.STRAIN_GAUGE_APPLICATION_TIME',
        'FATIGUE.SAMPLING_FREQUENCY',
        'FATIGUE.RECORDING_MODE',
        'FATIGUE.RECORDED_CHANNELS',
        'FATIGUE.PRELOAD_FORCE',
        'FATIGUE.BEACH_MARKS_ACTIVE',
        'FATIGUE.TERMINATION_REASON',
        'FATIGUE.CYCLES_GROSS',
        'FATIGUE.CYCLES_NET',
        'FATIGUE.CYCLES_TO_10_PERCENT_STRAIN_GAUGE_DROP',
    ],
    # reused standard types -> properties this notebook writes
    'EXPERIMENTAL_STEP': ['$NAME', 'START_DATE', 'FINISHED_FLAG', 'EXPERIMENTAL_STEP.EXPERIMENTAL_DESCRIPTION'],
    'GENERAL_PROTOCOL': ['$NAME', 'FOR_WHAT', 'PROCEDURE'],
    'INSTRUMENT': ['$NAME', 'MANUFACTURER'],
    'INSTRUMENT.CAMERA': ['$NAME', 'MANUFACTURER'],
    'INSTRUMENT.MEASURING_AMPLIFIER': ['$NAME', 'MANUFACTURER'],
    'TESTING_MACHINE': ['$NAME', 'MANUFACTURER', 'TESTING_MACHINE_DRIVE_TYPE', 'TESTING_MACHINE_LOAD_TYPE',
                        'MAX_LOAD_STATIC_PRIMARY', 'MAX_LOAD_DYNAMIC_PRIMARY'],
    'AUXILIARY_MATERIAL': ['$NAME', 'MANUFACTURER', 'AUXILIARY_MATERIAL_TYPE'],
}

# (type, property, term) - vocabulary terms (new vocabularies + EXTEND terms) that the workflow relies on
REQUIRED_TERMS = [
    ('SAMPLE.WELDED_FATIGUE_SPECIMEN', 'FATIGUE.NEXT_STEP', 'S01_PRE_QUALITY_CHECK'),
    ('SAMPLE.WELDED_FATIGUE_SPECIMEN', 'FATIGUE.NEXT_STEP', 'DONE'),
    ('SAMPLE.WELDED_FATIGUE_SPECIMEN', 'EXPERIMENTAL_STEP.WELDMENT_TYPE', 'WELDING_BUTT_WELD'),
    ('SAMPLE.WELDED_FATIGUE_SPECIMEN', 'EXPERIMENTAL_STEP.WELDMENT_TYPE', 'WELDING_CRUCIFORM_WELD'),
    ('SAMPLE.WELDED_FATIGUE_SPECIMEN', 'FATIGUE.TARGET_QUALITY_LEVEL', 'B'),
    ('SAMPLE.WELDED_FATIGUE_SPECIMEN', 'FATIGUE.ISO_5817_ANNEX_C_RATING', 'B90'),
    ('SAMPLE.WELDED_FATIGUE_SPECIMEN', 'FATIGUE.LOAD_LEVEL', 'HIGH'),
    ('EXPERIMENTAL_STEP.MECHANICAL_TEST.CYCLIC_FATIGUE_TEST', 'FATIGUE.RECORDING_MODE', 'PEAK_VALLEY'),
    ('EXPERIMENTAL_STEP.MECHANICAL_TEST.CYCLIC_FATIGUE_TEST', 'FATIGUE.TERMINATION_REASON', 'RUN_OUT'),
    ('AUXILIARY_MATERIAL', 'AUXILIARY_MATERIAL_TYPE', 'STRAIN_GAUGE'),
    ('AUXILIARY_MATERIAL', 'AUXILIARY_MATERIAL_TYPE', 'CRACK_DETECTION_AGENT'),
    ('TESTING_MACHINE', 'TESTING_MACHINE_DRIVE_TYPE', 'SERVOHYDRAULIC'),
]

problems = []
for t, codes in REQUIRED_TYPES.items():
    try:
        have = assigned_props(t)
    except Exception as e:  # unknown type -> pyBIS raises while loading the type
        problems.append(f"object type {t}: missing on instance ({e})")
        continue
    missing = [c for c in codes if c.lower() not in have]
    if missing:
        problems.append(f"object type {t}: properties not assigned: {', '.join(missing)}")

for t, prop, term in REQUIRED_TERMS:
    if t not in _ASSIGNED or prop.lower() not in _ASSIGNED[t]:
        continue  # already reported above
    probe = o.new_sample(type=t, space=SPACE)  # never saved
    try:
        probe.p[prop.lower()] = term  # pyBIS validates the term against the vocabulary
    except (KeyError, ValueError) as e:
        problems.append(f"vocabulary term {term} for {t}.{prop}: missing ({e})")

if problems:
    print("Masterdata incomplete - provision it first (bam-masterdata sync by an admin, or tools/openbis_apply.js):")
    for p in problems:
        print("  -", p)
    raise RuntimeError("Stopped: required masterdata missing (see list above). Nothing was written.")
print("Masterdata check passed.")

## 4. Series collections

The project `/FB_72/WELDED_FATIGUE` must already exist. One `COLLECTION` per test series; add further series here
(code scheme: `SERIES_<n>_<GEOMETRY>`, specimens `S<n><B|C>-NN`).

In [ ]:
SERIES = [
    {"code": "SERIES_1_BUTT_WELD", "name": "Series 1 - butt weld//Serie 1 - Stumpfnaht"},
    {"code": "SERIES_1_CRUCIFORM_WELD", "name": "Series 1 - cruciform weld//Serie 1 - Kreuzstoss"},
]
SPECIMEN_TYPE = "SAMPLE.WELDED_FATIGUE_SPECIMEN"

for c in SERIES:
    ensure_collection(PROJECT_ID, c["code"], c["name"], SPECIMEN_TYPE)

## 5. Inventory objects (created once, then only linked)

Mandatory properties learned on dev1: `INSTRUMENT*` need `MANUFACTURER`; `TESTING_MACHINE` needs `MANUFACTURER`,
`TESTING_MACHINE_DRIVE_TYPE`, `TESTING_MACHINE_LOAD_TYPE`, `MAX_LOAD_STATIC_PRIMARY`, `MAX_LOAD_DYNAMIC_PRIMARY`.
Values marked *to be completed* / *placeholder* are **placeholders - correct them before the first run**.
The inventory collections in `/MATERIALS` must exist (they are not created here). The caliper and the GOM software are
deliberately **not** inventory objects (named in the workflow plan / SCAN step text only).

In [ ]:
INVENTORY = {
    'GOM': {"code": 'GOM_ATOS_5', "type": 'INSTRUMENT', "collection": '/MATERIALS/7_2_EQUIPMENT/INSTRUMENTS',
        "props": {"$NAME": "ZEISS GOM ATOS 5", "MANUFACTURER": "ZEISS", "DEVICE_MODEL_NAME": "ATOS 5", "DESCRIPTION": "Fringe-projection 3D scanner, GOM Inspect software for weld analysis//Streifenprojektions-3D-Scanner, Nahtanalyse mit GOM-Software"}},
    'MICROSCOPE': {"code": 'DIGITAL_MICROSCOPE_01', "type": 'INSTRUMENT', "collection": '/MATERIALS/7_2_EQUIPMENT/INSTRUMENTS',
        "props": {"$NAME": "Digital microscope//Digitalmikroskop", "DESCRIPTION": "DEMO - model to be completed//DEMO - Modell ergänzen", "MANUFACTURER": "to be completed (DEMO)"}},
    'LUXONIS': {"code": 'LUXONIS_OAK_4D_PRO_CAM', "type": 'INSTRUMENT.CAMERA', "collection": '/MATERIALS/7_2_EQUIPMENT/INSTRUMENTS',
        "props": {"$NAME": "Luxonis OAK-4D-Pro camera system", "MANUFACTURER": "Luxonis", "DEVICE_MODEL_NAME": "OAK-4D-Pro"}},
    'MACRO': {"code": 'MACRO_CAMERA_01', "type": 'INSTRUMENT.CAMERA', "collection": '/MATERIALS/7_2_EQUIPMENT/INSTRUMENTS',
        "props": {"$NAME": "Macro camera//Makro-Kamera", "DESCRIPTION": "DEMO - model to be completed//DEMO - Modell ergänzen", "MANUFACTURER": "to be completed (DEMO)"}},
    'IMC': {"code": 'IMC_AMPLIFIER_01', "type": 'INSTRUMENT.MEASURING_AMPLIFIER', "collection": '/MATERIALS/7_2_EQUIPMENT/INSTRUMENTS',
        "props": {"$NAME": "IMC measuring amplifier//IMC-Messverstärker", "MANUFACTURER": "imc Test & Measurement", "DESCRIPTION": "DEMO - model to be completed//DEMO - Modell ergänzen"}},
    'RIG': {"code": 'SERVOHYDRAULIC_RIG_01', "type": 'TESTING_MACHINE', "collection": '/MATERIALS/7_2_EQUIPMENT/TESTING_MACHINES',
        "props": {"$NAME": "Servo-hydraulic testing machine//Servohydraulische Prüfmaschine", "TESTING_MACHINE_DRIVE_TYPE": "SERVOHYDRAULIC", "TESTING_MACHINE_LOAD_TYPE": "AXIAL", "DESCRIPTION": "DEMO - max. loads are placeholders (100 kN), model to be completed//DEMO - Maximallasten sind Platzhalter (100 kN), Modell ergänzen", "MANUFACTURER": "to be completed (DEMO)", "MAX_LOAD_STATIC_PRIMARY": "100", "MAX_LOAD_DYNAMIC_PRIMARY": "100"}},
    'DMS': {"code": 'DMS_HBM_LY11_6_120', "type": 'AUXILIARY_MATERIAL', "collection": '/MATERIALS/7_2_EQUIPMENT/STRAIN_GAUGES',
        "props": {"$NAME": "Strain gauge HBM LY11-6/120 (120 Ohm)", "MANUFACTURER": "HBM", "AUXILIARY_MATERIAL_TYPE": "STRAIN_GAUGE"}},
    'X60': {"code": 'ADHESIVE_HBM_X60', "type": 'AUXILIARY_MATERIAL', "collection": '/MATERIALS/7_2_MATERIALS/CONSUMABLES',
        "props": {"$NAME": "Adhesive HBM X60//Klebstoff HBM X60", "MANUFACTURER": "HBM", "AUXILIARY_MATERIAL_TYPE": "ADHESIVE"}},
    'LUM': {"code": 'CRACK_LUMINESCENCE_SET', "type": 'AUXILIARY_MATERIAL', "collection": '/MATERIALS/7_2_MATERIALS/CONSUMABLES',
        "props": {"$NAME": "Crack luminescence set//Risslumineszenz-Set", "MANUFACTURER": "MR Chemie GmbH", "AUXILIARY_MATERIAL_TYPE": "CRACK_DETECTION_AGENT"}},
}

# step kind -> inventory keys linked as parents of that step object
STEP_PARENTS = {"SCAN": ["GOM"], "TEST": ["DMS", "X60", "LUM", "LUXONIS", "IMC", "RIG"], "FRACTURE": ["MACRO", "MICROSCOPE"]}


def inv_id(key):
    x = INVENTORY[key]
    _, space, project, _ = x["collection"].split("/")
    return f"/{space}/{project}/{x['code']}"


for key, x in INVENTORY.items():
    _, space, project, coll = x["collection"].split("/")
    if _get_or_none(o.get_experiment, x["collection"]) is None:
        raise RuntimeError(f"Inventory collection {x['collection']} missing - create it in the ELN inventory first.")
    ensure_sample(inv_id(key), x["type"], experiment=x["collection"], props=x["props"])

## 6. Workflow plan (GENERAL_PROTOCOL in the project)

The workflow plan is a method description, so it is a `GENERAL_PROTOCOL` (the `SOP` type on the instance requires an
E-Akte ID/link). Its `PROCEDURE` text is read from `generated/welded_fatigue_v2/sop_workflow_plan.txt` (step -> location table).

In [ ]:
WORKFLOW_PLAN_ID = f"{PROJECT_ID}/WORKFLOW_PLAN"
plan_text = (DESIGN_DIR / "sop_workflow_plan.txt").read_text(encoding="utf-8")

ensure_sample(WORKFLOW_PLAN_ID, "GENERAL_PROTOCOL", props={
    "$NAME": "Workflow plan S-N tests on welded joints//Ablaufplan Wöhlerversuche Schweißverbindungen",
    "FOR_WHAT": "Where each step of the S-N workflow is documented//Wo jeder Schritt des Wöhler-Ablaufs dokumentiert wird",
    "PROCEDURE": plan_text,
})

## 7. Register a specimen and start its step objects

* `register_specimen(code, collection, props)` - creates the specimen in its series collection (`$NAME` = code,
  `FATIGUE.NEXT_STEP` defaults to `S01_PRE_QUALITY_CHECK`).
* `start_step(specimen_code, kind)` - `kind` in `SCAN` (03-04), `TEST` (07-11), `FRACTURE` (12). Creates
  `<code>-<kind>` in the specimen's collection with parents = specimen + inventory (`STEP_PARENTS`) and sets
  **Next step** on the specimen to the first step the object covers.
* `finish_step(specimen_code, kind, next_step=None)` - sets `FINISHED_FLAG` on the step object and advances **Next step**
  (SCAN -> S05, TEST -> S12, FRACTURE -> DONE; pass `next_step="DONE"` e.g. for a run-out without fracture analysis).
* `set_next_step(specimen_code, term)` - for the specimen-only steps 01, 02, 05, 06.

Operators see "what's next" in the series table (column *Next step*). Measured values of the steps are filled by the parser
or in the ELN form - the step objects created here are the containers.

In [ ]:
STEP_DEF = {
    "SCAN": {"type": "EXPERIMENTAL_STEP", "name": "03-04 3D scan & weld analysis",
             "desc": "ZEISS GOM ATOS 5 scan; weld analysis in GOM software, rating stored on the specimen (section 04)"
                     "//3D-Scan mit ZEISS GOM ATOS 5; Nahtanalyse in der GOM-Software, Bewertung an der Probe (Sektion 04)",
             "next_on_start": "S03_3D_SCAN", "next_on_finish": "S05_SERIES_ASSIGNMENT"},
    "TEST": {"type": "EXPERIMENTAL_STEP.MECHANICAL_TEST.CYCLIC_FATIGUE_TEST", "name": "07-11 Cyclic fatigue test",
             "desc": None,
             "next_on_start": "S07_STRAIN_GAUGE_APPLICATION", "next_on_finish": "S12_FRACTURE_SURFACE_ANALYSIS"},
    "FRACTURE": {"type": "EXPERIMENTAL_STEP", "name": "12 Fracture surface analysis",
                 "desc": "Macroscopic and light-microscopic fracture surface analysis"
                         "//Makroskopische und lichtmikroskopische Bruchflächenanalyse",
                 "next_on_start": "S12_FRACTURE_SURFACE_ANALYSIS", "next_on_finish": "DONE"},
}


def specimen_id(code):
    return f"{PROJECT_ID}/{code}"


def set_next_step(specimen_code, term):
    s = o.get_sample(specimen_id(specimen_code))
    s.p["fatigue.next_step"] = term
    s.save()
    print(f"  > {specimen_code}: Next step = {term}")


def register_specimen(code, collection, props=None):
    """Create specimen <code> in series collection <collection> (code of the collection, e.g. 'SERIES_1_BUTT_WELD')."""
    p = {"$NAME": code, "FATIGUE.NEXT_STEP": "S01_PRE_QUALITY_CHECK", **(props or {})}
    return ensure_sample(specimen_id(code), SPECIMEN_TYPE, experiment=f"{PROJECT_ID}/{collection}", props=p)


def start_step(specimen_code, kind, props=None):
    """Create <specimen>-<kind> (SCAN/TEST/FRACTURE) with specimen + inventory parents; update Next step."""
    d = STEP_DEF[kind]
    spec = o.get_sample(specimen_id(specimen_code))
    step_id = specimen_id(f"{specimen_code}-{kind}")
    if _get_or_none(o.get_sample, step_id) is not None:
        print(f"  = {step_id} (exists) - Next step unchanged")
        return o.get_sample(step_id)
    p = {"$NAME": f"{specimen_code} · {d['name']}",
         "START_DATE": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
         "FINISHED_FLAG": False,
         "EXPERIMENTAL_STEP.EXPERIMENTAL_DESCRIPTION": d["desc"],
         **(props or {})}
    exp_id = getattr(spec.experiment, "identifier", spec.experiment)
    step = ensure_sample(step_id, d["type"], experiment=exp_id,
                         parents=[spec.identifier] + [inv_id(k) for k in STEP_PARENTS[kind]], props=p)
    set_next_step(specimen_code, d["next_on_start"])
    return step


def finish_step(specimen_code, kind, next_step=None):
    step = o.get_sample(specimen_id(f"{specimen_code}-{kind}"))
    step.p["finished_flag"] = True
    step.save()
    set_next_step(specimen_code, next_step or STEP_DEF[kind]["next_on_finish"])

## 8. Example: new specimen S1B-07 (guarded)

Set `RUN_EXAMPLE = True` to register S1B-07 and start its SCAN step. Values are illustrative - replace them with the real ones.

In [ ]:
RUN_EXAMPLE = False

if RUN_EXAMPLE:
    register_specimen("S1B-07", "SERIES_1_BUTT_WELD", {
        "ALIAS": "22_1",                                   # internal BAM designation
        "FATIGUE.SOURCE_SHEET": "22",
        "EXPERIMENTAL_STEP.WELDMENT_TYPE": "WELDING_BUTT_WELD",
        "FATIGUE.TARGET_QUALITY_LEVEL": "B",
        # 01 Pre-quality check
        "FATIGUE.INITIAL_ANGULAR_DISTORTION": 0.48,
        "FATIGUE.INITIAL_EDGE_MISALIGNMENT": 0.27,
        "FATIGUE.WAS_STRAIGHTENED": False,
    })
    start_step("S1B-07", "SCAN")   # creates S1B-07-SCAN <- specimen + ZEISS GOM ATOS 5; Next step = S03_3D_SCAN
    # later: finish_step("S1B-07", "SCAN"); start_step("S1B-07", "TEST"); finish_step("S1B-07", "TEST"); ...

## 9. Dataset upload helper

`DATASET_MAPPING` is the `dataset_mapping` table of `design.json` (file kind -> standard dataset type + target object).
Only standard dataset types are used; verify they exist on the instance before the first upload.
Routine uploads of IMC/GOM files after each run belong to the parser path; this helper covers manual uploads.

In [ ]:
DATASET_MAPPING = {
    'specimen_photos': ('RAW_DATA', 'SPECIMEN', '01'),  # Specimen photos
    'post_processing_report': ('ATTACHMENT', 'SPECIMEN', '02'),  # Post-processing report
    'scan_raw_data': ('RAW_DATA', 'SCAN', '03'),  # 3D scan raw data
    'gom_evaluation_export': ('ANALYZED_DATA', 'SCAN', '04'),  # Evaluation export
    'randomisation_script': ('SOURCE_CODE', 'COLLECTION', '05'),  # Randomisation script
    'strain_gauge_position_drawing': ('ATTACHMENT', 'TEST', '07'),  # Strain gauge position drawing
    'setup_photo': ('RAW_DATA', 'TEST', '07'),  # Setup photo
    'bending_moment_raw_data': ('RAW_DATA', 'TEST', '09'),  # Bending moment raw data
    'load_programme': ('MEASUREMENT_PROTOCOL_FILE', 'TEST', '10'),  # Load programme
    'measurement_raw_data': ('RAW_DATA', 'TEST', '10'),  # Measurement raw data
    'monitoring_images': ('RAW_DATA', 'TEST', '10'),  # Monitoring images (time-lapse)
    'cleaned_measurement_data': ('PROCESSED_DATA', 'TEST', '11'),  # Cleaned measurement data
    'evaluation_plots': ('FIGURE.PLOT', 'TEST', '11'),  # Evaluation plots
    'conversion_script': ('SOURCE_CODE', 'TEST', '11'),  # Conversion script
    'macro_photos': ('RAW_DATA', 'FRACTURE', '12'),  # Macro photos
    'microscope_images_crack_origin': ('RAW_DATA', 'FRACTURE', '12'),  # Microscope images of crack origin
    'beach_mark_evaluation': ('ANALYZED_DATA', 'FRACTURE', '12'),  # Beach-mark evaluation
}


def upload(kind, files, specimen_code=None, collection=None):
    """Upload local files as a dataset of the mapped type to the mapped object.
    kind: key of DATASET_MAPPING; specimen_code for specimen/step targets; collection code for 'COLLECTION' targets."""
    ds_type, target, step = DATASET_MAPPING[kind]
    missing = [f for f in files if not Path(f).is_file()]
    if missing:
        raise FileNotFoundError(f"not found: {missing}")
    if target == "COLLECTION":
        if not collection:
            raise ValueError(f"{kind}: pass collection='SERIES_...'")
        ds = o.new_dataset(type=ds_type, experiment=f"{PROJECT_ID}/{collection}", files=[str(f) for f in files])
        where = f"{PROJECT_ID}/{collection}"
    else:
        if not specimen_code:
            raise ValueError(f"{kind}: pass specimen_code='S1B-..'")
        where = specimen_id(specimen_code if target == "SPECIMEN" else f"{specimen_code}-{target}")
        if _get_or_none(o.get_sample, where) is None:
            raise RuntimeError(f"{where} does not exist - run register_specimen()/start_step() first")
        ds = o.new_dataset(type=ds_type, sample=where, files=[str(f) for f in files])
    ds.save()
    print(f"  + dataset {ds.permId} ({ds_type}, step {step}) -> {where}")
    return ds


# example (manual):
# upload("load_programme", ["C:/data/S1B-07/load_programme.xlsx"], specimen_code="S1B-07")
# upload("randomisation_script", ["randomise_series1.py"], collection="SERIES_1_BUTT_WELD")

## 10. Close

Logging out ends the session token in this kernel; the PAT itself stays valid for the next run.

In [ ]:
# o.logout()